# 02 · 질환 연관 분석 — 로지스틱 회귀와 오즈비

특정 질환을 고르지 않고, 환자 100명 이상인 ICD-10 3자리 코드 전부에 같은 절차를 적용한다.
가설을 세워 검증하는 것이 아니라, 파형의 어느 부분이 어느 질환과 함께 움직이는지를 한 번에 훑는다.

## 모형

```
logit P(D_p = 1) = b0 + b · z(X_p)
```

- `D_p ∈ {0,1}` 해당 ICD 코드 보유 여부. 대조군은 그 코드가 없는 나머지 전원
- `X_p` 환자 p의 요약 특징(박동별 중앙값), 코호트 전체에서 z-표준화
- `exp(b)` **특징이 1 표준편차 높을 때 질환 오즈가 몇 배가 되는가** = 효과 크기

**공변량은 넣지 않는다.** 파형 특징 자체와 질환의 연관을 보는 것이 목적이다.

**Firth 보정은 쓰지 않는다.** 사례 최소 코드(n=100~103)에서도 계수 변화가 0.0~2.8%에 그쳤다.
사건당 변수 수가 14라 보정이 필요한 구간이 아니다.

## 선행 연구는 어떻게 하는가

| 연구 | 환자 요약 | 검정 | 효과 크기 | 다중검정 |
|---|---|---|---|---|
| ECG PheWAS (EHJ Digital Health 2025) | 요약 안 함 | Mann–Whitney U | AUROC | Bonferroni ~8e−6 |
| ECG 비지도 표현 (npj Digital Medicine 2025) | 잠재공간 점수 1개 | 로지스틱 | 오즈비 | Bonferroni 3.1e−5 |
| AnyPPG (arXiv 2025-11) | 입원당 세그먼트 20개 | 다중라벨 분류 | AUC | 명시 없음 |
| dicrotic notch (UK Biobank 148,310명) | CNN 점수 1개 | 로지스틱 · Cox | SD당 오즈비 | Bonferroni |
| **본 연구** | **박동별 중앙값 43개** | **로지스틱** | **오즈비** | **Bonferroni 6.8e−6** |

선행은 대부분 **점수 하나**를 환자값으로 쓴다. 그림의 행이 1개다. 우리는 행이 43개라
어느 파형 특징이 움직이는지가 보이지만, 검정 수가 43배다. 다중검정 기준은 선행 세 편이
모두 쓰는 Bonferroni를 그대로 따랐다.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import numpy as np
import pandas as pd

from ppg_fm import paths, features as F

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)
print("프로젝트", paths.ROOT)
print("데이터  ", paths.data_root())
from ppg_fm.report import Report
rep = Report("02_logistic/02_association_figure2")
print("산출물 →", rep.dir)
from IPython.display import Image

In [ ]:
from ppg_fm.analysis import logistic as L, figures as G

P, M, codes = L.load_cohort(min_cases=100)
print(f"환자 {len(P):,} · 질환 {len(codes)} · 특징 {len(F.PATIENT)}")
print(f"검정 수 {len(codes)*len(F.PATIENT):,} · Bonferroni 임계 {L.bonferroni(len(codes)*len(F.PATIENT)):.2e}")

## 1. 회귀 — 7,396 셀

In [ ]:
out = paths.reports("figure2_or.csv")
if out.exists():
    R = pd.read_csv(out)
else:
    R = L.run(min_cases=100)          # 수 분 소요
summ = pd.Series(L.summarize(R)).rename("값").rename_axis("항목").reset_index()
rep.table(R, "figure2_or.csv", "질환 × 특징 오즈비 전수")
rep.table(summ, "figure2_summary.csv", "검정 수 · 유의 셀 요약")
summ

## 2. Figure 2

In [ ]:
png = G.figure2(R, out=rep.path("figure2_or.png"))
rep.adopt(png, "Figure 2 — 진단코드 × 파형 특징")
from IPython.display import Image
Image(str(png))

### 읽는 법

- 색의 **방향**이 먼저다 — 붉은색은 오즈비 > 1(특징이 높을수록 질환), 파란색은 < 1
- 색의 **진하기**는 효과 크기다. 비워진 칸은 Bonferroni를 통과하지 못한 것이지 «차이 없음»의 증거가 아니다
- 세로 띠는 **한 질환이 여러 특징에 걸쳐** 나타난다는 뜻, 가로 띠는 **한 특징이 여러 질환에서** 움직인다는 뜻

## 3. 유의 특징이 넓게 퍼진 질환

In [ ]:
br = L.breadth(R, n=15)
rep.table(L.breadth(R, n=10**6), "significant_breadth.csv", "질환별 유의 특징 수")
br

## 4. 효과가 큰 셀

In [ ]:
tc = L.top_cells(R, n=15)
rep.table(L.top_cells(R, n=10**6), "significant_cells.csv", "Bonferroni 통과 셀 전수")
tc.round(4)

## 5. 결과를 읽을 때 같이 볼 것

효과가 가장 큰 셀 상위권에 **외상(S27 외상성 흉부손상 · S22 늑골골절)과 출혈(I60 · I61)**이 올라온다.
이 환자군은 대조군보다 젊고 심박수가 빠르다 — S27은 평균 14세 젊고 HR이 7 bpm 빠르다.
같은 셀의 계수는 무보정 −0.665에서 연령을 넣으면 −0.333, 전체 보정에서 −0.087로 유의성을 잃는다.

반면 **넓게 퍼진 질환**(I50 심부전 · I48 심방세동 · R57 쇼크 · N18 만성신질환 · I25 만성허혈성심질환)은
보정 전후로 계수가 크게 흔들리지 않는다. 두 층위를 구분해서 읽어야 한다.

In [ ]:
abl = paths.reports("covariate_ablation.csv")
if abl.exists():
    A = pd.read_csv(abl)
    t = A[A.icd10.isin(["S27", "I50", "I48"])].pivot_table(
        index=["icd10", "feat"], columns="model", values="beta")
    t["변화폭"] = (t.iloc[:, 0].abs() - t.iloc[:, -1].abs())
    display(t.reindex(t["변화폭"].abs().sort_values(ascending=False).index).head(12).round(3))

## 6. Figure 2 변형 — 같은 자료를 다르게 자른 판

논문에서 어느 판을 본문에 쓰고 어느 판을 보충으로 돌릴지는 결과를 보고 정한다.

### 6-1. 신호가 있는 것만 — compact

유의 특징이 5개 이상인 질환 54개만 남기고 질환명을 붙였다. 전체 판은 172개 코드를 다 보여주지만
셀이 작아 읽기 어렵다. 이 판은 실제로 신호가 나온 부분만 크게 본다.

In [ ]:
png = G.figure2_compact(R, min_sig=5, out=rep.path("figure2_compact.png"))
rep.adopt(png, "유의 특징 ≥5인 질환 54개 × 특징 43개")
Image(str(png))

### 6-2. 순환계 확대

파형 특징이 혈역학을 반영한다면 먼저 나타나야 할 자리다. I 코드만 떼어 질환명과 함께 본다.

In [ ]:
png = G.figure2_chapter(R, "Circulatory", out=rep.path("figure2_circulatory.png"))
rep.adopt(png, "순환계 ICD 코드 확대")
Image(str(png))

### 6-3. ICD 장별 소패널

계통마다 어떤 특징 계열이 움직이는지 나란히 놓고 비교한다.

In [ ]:
png = G.figure2_panels(R, out=rep.path("figure2_panels.png"))
rep.adopt(png, "ICD 장별 소패널")
Image(str(png))

### 6-4. 효과가 큰 셀 — 값으로 읽는 판

히트맵은 방향과 패턴을 보여주지만 값 자체는 읽기 어렵다. 상위 40개를 오즈비 축에 놓는다.

In [ ]:
png = G.figure2_top_cells(R, n=40, out=rep.path("figure2_top_cells.png"))
rep.adopt(png, "효과 상위 40셀 오즈비")
Image(str(png))

### 6-5. 특징 쪽에서 본 판

어느 특징이 몇 개 질환에서, 어느 방향으로 움직이는가. Figure 2의 우측 막대를 방향까지 나눠 펼친 것이다.

In [ ]:
png = G.figure2_feature_breadth(R, out=rep.path("figure2_feature_breadth.png"))
rep.adopt(png, "특징별 유의 질환 수 (방향 구분)")
Image(str(png))

## 산출물

In [ ]:
rep.done("로지스틱 회귀 · 오즈비 · Figure 2")
rep.summary()